### **1. 请简述 Spark 的核心组件及其作用？**
答：简单来说，RDD 就是 Spark 中最基本的分布式内存数据抽象。
它有三个核心特点，第一是不可变：一旦创建就不能修改，只能通过转换生成新的 RDD。二是可分区：数据被切分成多份，分布在集群不同节点上并行计算。三是可容错：通过记录血缘关系，如果数据丢失可以自动恢复。
大家都知道编程语言中都有各自的数据结构，比如python中的list，java中的arrayList，Spark中的RDD和这个语言的数据结构类似，不过它在这些数据结构之上做了更高级的封装，使其具备分区，容错等属性。

### **2. Spark 中 RDD、DataFrame 和 Dataset 的区别是什么？**
首先是RDD，它是最底层的抽象，是分布式的 Java/Scala 对象集合。优点是细粒度控制，缺点是缺乏元数据信息，无法进行语义优化。
然后是DataFrame，它类似关系型数据库的表，带有 列名和类型。基于 Catalyst 优化器，执行效率通常高于 RDD。
最后是Dataset，它是DataFrame 的扩展，提供了类型安全的对象操作接口。在 Scala/Java 中常用，结合了 RDD 的类型安全和 DataFrame 的执行优化。

### **3. 什么是 DAG？Spark 为什么使用 DAG 而不是像 MapReduce 那样直接执行？**
DAG是有向无环图，Spark将作业转换为一系列阶段，每个阶段包含多个含依赖关系的task，形成DAG，之所以Spark使用DAG是因为和MapReduce不一样的是它是内存计算，大部分任务都可以在内存中直接计算，迭代计算速度非常快
然后是优化执行，dag调度去可以在运行前进行全局优化，如流水线执行、分区裁剪等，再然后是可以基于血缘进行容错，而非单纯依赖数据副本。

### **4. 请描述 Spark 任务的提交流程（从 submit 到执行）**
首先用户先提交spark-submit，然后driver启动，构建上下文，driver向k8s申请资源，然后k8s启动executor，driver将job按宽依赖划分成阶段，driver再将阶段划分成task，并分发给executor，
executor执行任务，并将结果返回给Driver或写入存储。

### **5. 什么是宽依赖和窄依赖？它们对 Stage 划分有什么影响？**
窄依赖是父RDD的一个分区只被子RDD的一个分区使用（如map，filter，union）。允许流水线执行。
宽依赖是父RDD的一个分区被子RDD的多个分区使用（如groupByKey，reduceByKey，Join）。涉及shuffle操作。
stage划分的原则是遇到宽依赖是会切断，生成新的stage，窄依赖则合并到同一个stage中。

### **6. 请说说Spark的Shuffle机制演进与调优**
答：Spark的Shuffle实现随着版本的迭代发生了多次重大变更，主要目标是提高性能、减少内存占用并增强稳定性。
早期Spark用的是Hash Shuffle，每个MapTask 会为每个ReduceTask生成一个单独的文件，优点是不需要排序，会快一点，缺点是需要同时打开大量的缓冲区，虽然后来引入了一些机制合并文件，但依旧有些力不从心；
第二个阶段是Sort Shuffle（Spark2.0之后成为默认选项），通过每个MapTask最终只产生一个数据文件和一个索引文件解决了小文件问题，原理是数据先写入一个内存缓冲区中，当缓冲区满时，对数据进行排序并溢写到磁盘，最后将所有溢写文件合并成一个大的数据文件和一个索引文件，Reduce端通过索引文件去数据文件中读取属于自己的数据，优点是文件数量大幅减少，且支持在内存排序，有利于后续的聚合和排序操作，缺点是引入了排序，CPU开销略高于Hash Shuffle。
第三个阶段是Tungsten-Sort，这是对Sort Shuffle的深度优化，主要利用堆外内存和二进制格式处理数据，优点是可以大幅减少JVM的内存占用和CPU开销，缺点是有触发条件，大概就是需要使用Spark SQL自带的序列化器和KryoSeializer，且没有聚合操作。
至于Shuffle的调优，核心思想就是减少Shuffle数据量或者减少Shuffle次数或者优化Shuffle过程的资源使用.
你可以先过滤掉不必要的数据，其次在Join操作中，如果一个表比较小，可以使用广播变量将小表广播到所有Executor，这样可以避免shuffle。
其次就是参数的调优了，并行度过高会导致大量的小文件和调度开销，过低会导致单个Task处理数据量过大，一般设置为总核数的2-3倍即可。
内存缓冲区默认是32KB，如果有足够的内存，可以适当调大，减少磁盘溢写次数，reduce端拉取数据时也有个内存缓冲区大小，调大它可以减少网络拉取次数，但会增加内存压力
也可以开启shuffle文件压缩，虽然会增加cpu负担，但能显著减少磁盘和网络io
还有就是对数据倾斜的调优，大致可以通过提升并行度，广播小表，给倾斜key加盐值的方式减小shuffle的压力

### **7. 在数据倾斜场景下，除了常规的加盐（Salting），如何利用 Spark 3.x 的 AQE (Adaptive Query Execution) 动态合并小分区或动态调整倾斜连接策略？请描述其内部触发机制。**
Spark的AQE是Spark引入的新特性嘛，它利用运行时的统计信息动态优化查询计划，从而解决了传统Spark SQL在编译阶段无法感知数据分布的痛点。原理主要是通过动态合并Shuffle分区，及动态处理倾斜Join来做的。
首先是动态合并Shuffle分区，在没有AQE的情况下，我们需要手动设置分区数，如果设置得太大，会产生大量的小文件，导致大量随机小IO和网络开销；如果设置得太小，单个Task处理数据量过大，容易OOM，引入AQE以后，AQE会在Shuffle阶段结束后，根据实际运行时的数据分布，动态地将相邻的小分区合并成较大的分区，以减少task数量，至于原理，是spark在driver端收集了每个shuffle map stage输出的各个分区的统计信息，根据AQE中设置的参数来确定合并后的目标大小，内部走的是一个判断逻辑，首先从第一个分区开始遍历，一旦大小接近参数值，就将其标记成一个合并后的新分区，然后就是如果单个分区大小本身就超过了目标大小，它会被保留为一个独立的分区，最后是累加后的总和远小于目标大小，但已经到了最后一个分区，也会强制合并，然后就是根据合并后的分区映射关系，动态重写后续stage的物理执行计划。
然后是动态处理倾斜key，这是AQE针对数据倾斜的方案，在没有AQE的情况下，我们需要手动编写加盐值的逻辑，非常繁琐而且还容易出错，AQE可以在运行时自动检测倾斜并拆分处理，原理就是，AQE可以识别到分区数据量是不是大于了配置的参数，一旦检测到倾斜分区，它就会根据比例将大的分区拆成N个小分区，把非倾斜侧的需要Join的分区复制N份，广播拆分的小分区中，再执行Join

### **8. 如果开启 AQE 后，某些作业反而变慢了，可能的原因是什么？如何排查？**
开启AQE后作业变慢，通常是因 AQE的动态优化策略与实际的数据分布、集群资源状态或代码逻辑发生了冲突。虽然AQE在大多数情况下能提升性能，但在特定场景下，它的副作用可能会超过其带来的收益。
一些常见原因有动态合并shuffle分区导致并行度下降，导致作业运行时间变长，CPU空闲，然后是动态倾斜Join的拆分开销过大，他会复制非倾斜侧的分区数据嘛，如果复制得比较多也会产生大量的网络传输和磁盘io，然后就是运行时统计信息的收集和规划延迟，再stage比较多的情况下，每个stage都会由driver收集统计信息，累计的调度延时也不可忽视。
排查思路肯定就是检查Spark UI嘛，你可以找到哪个页面的stage，观察一下被AQE修改的部分，观察shuffle数据量是否异常，task的分布是否不均，如果所有task都变慢了，可能是被过度合并了，如果是长尾问题，可能是倾斜处理失效或引入了新的倾斜，如果shuffle数据异常，意味着join的策略选择错误，本来该广播的却选择了shuffle。
然后就是调整AQE参数，尝试让其恢复正常，分区合并的话就要尝试让其合并后的分区数达到最佳实践的数量，或者调整分区大小以保留更多分区，然后就是手动调整join策略，直到恢复正常为止，还有一个点就是要检查driver的资源，如果driver资源紧张，可能会导致规划阶段耗时过长

### **9. 解释 Spark 统一内存管理模型中 Storage Memory 和 Execution Memory 的动态借用机制。**
Spark在1.6版本后引入了统一内存管理模型，旨在解决以前静态内存划分导致资源利用率不高的问题，在统一内存管理模型中，存储内存和和执行内存共享同一块堆内内存区域，这两块区域不是静态固定的，而是可以相互借用的，首先，这个

### **10. 当发生 Spill to Disk 时，对性能的具体影响有哪些？在什么配置下会优先选择抛 OutOfMemoryError 而不是 Spill？**
答：我先说一下溢写到磁盘对性能的影响，溢写的本质是将高速的内存读写转化为低速的磁盘io，
首先就是磁盘io开销加剧，spark虽然在写的时候会尽量顺序写入文件，但在shuffle read阶段，reduce需要从多个map端拉取文件，这不可避免的引入了随机io，比单纯的内存交换慢几个数量级，其次时数据在写入时需要序列化，读取时需要反序列化，对于复杂对象，这个过程非常消耗cpu
然后就是额外的排序与合并开销，当内存缓冲区满了触发溢写时，spark还会对内存中的数据进行排序，如果代码逻辑里面还有排序，还需要再内部进行二次排序，然后在task结束前，如果产生了多个溢写文件，spark还需要将这些文件和内存中的剩余数据一起进行归并排序，合并成一个大文件，文件越多，开销越大。
然后就是会拉长GC的时间，在溢写的过程中，为了管理磁盘上的数据指针，spark会频繁的创建辅助对象，频繁的内存申请和释放会导致堆内存碎片化，导致频繁GC，拖慢整个作业。
让然后就是溢写的数据是通过网络进行交换的，会导致网络请求的次数会增加。

然后我再说一下什么情况下会优先OOM而不是溢写，通过情况下，堆外内存溢出是最常见的，堆外内存不受JVM管理，且通常不直接支持像堆内内存那样灵活的溢写机制，就会OOM
然后是算子逻辑导致的无法spill，有些特定的数据结构不支持溢写，或者代价极高，spark容易崩溃，比如说广播变量，广播变量设计初衷就是希望能完全驻留内存的，溢写到磁盘就违背了初衷了，
然后就是加载数据那里，数据是放在堆内存的局部变量中，不属于spark统一内存管理的范畴，spark的内存管理器无法感知这部分内存，也就没法溢写了

### **11. 请描述 Catalyst 优化器的四个阶段（Analysis, Logical Optimization, Physical Planning, Code Generation）。**
第一个阶段，分析阶段，就是解析sql中的引用，就是利用spark的catalog来查找表名、列名，它会生成一个解析后的逻辑计划，如果在这个过程中发现引用不存在或者类型不匹配，查询会直接报错；
第二个阶段，逻辑优化阶段，核心是生成一个优化后的逻辑计划，这个阶段不涉及具体的物理实现（比如说用Hash Join还是Sort Merge Join），只关注逻辑上的等价变换，具体就是运用一些规则，比如说谓词下推，列剪枝，常量折叠，布尔简化，消除冗余之类的操作；
第三个阶段，物理计划阶段，将逻辑计划转化为物理上怎么做，一个逻辑计划一般会对应多个物理计划，Catalyst会根据策略选择一个最佳的物理计划，具体就是选择算子具体的实现方式，比如说选择join的方式。
第四个阶段就是代码生成阶段，这就是Spark SQL性能优于hive的关键，通过迭代器模式处理数据，这里就会使用全阶段代码生成，将多个物理算子合并成一个单一的java函数，同时在运行时动态生成java字节码，具体就是把物理计划树压成一个个RDD操作。

### **12. Whole-Stage Code Generation 是如何工作的？它解决了传统火山模型（Volcano Model）的什么瓶颈？**
首先我说说传统的火山模型嘛，火山模型的工作方式会把查询计划表示为一个算子树，每个算子都实现了一个统一的接口，比如说open()，next()，close等，父算子就调用子算子的next()方法获取一行数据，子算子处理完数据后返回给父算子，父算子处理改行数据，再次调用next()，如此往复，优点是简介易扩展，缺点就是在性能上存在严重的缺陷，主要源于虚函数调用和内存访问。
首先虚函数调用的开销来源于每处理一行数据，都需要在多个算子之间进行函数调用，如果查询计划有N层深度，处理一行数据就需要N次函数调用，在JVM中，这些调用无法被编译器有效管理，导致CPU指令流水线频繁中断
然后就是CPU缓存局部性差，数据在各个算子的函数栈中传递，导致CPU缓存命中率低
然后就是全阶段代码生成的原理，它的核心机制就是算子融合，它不会为每个算子单独生成代码，而是寻找物理计划中适合融合的算子链，将它们压扁成一个大的代码块。

### **13. 如果在执行计划中看到 Exchange 算子非常多，通常意味着什么？如何通过 repartition 或 coalesce 优化？**
Exchange算子多就说明在频繁的shuffle，会导致大量的网络传输，也会加重内存和磁盘压力。
repartition 或 coalesce 的核心优化目标肯定就是减少不必要的Shuffle，或者合并小分区以提高效率嘛
首先repartition会触发shuffle，当出现数据倾斜时，可以增加分区数，把数据再打散一点，然后就是可以提高并行度，如果再join之前按照join的key进行了repartition，也可以避免Spark在join时自动进行额外的shuffle，或者至少让shuffle按照预期进行。
coalesce不会触发shuffle，用于减少分区数，适用于那种过滤了大量数据，但分区数没变，导致产生了大量小文件或空任务，此时就能使用coalesce减少分区数，还有就是在写入数据时，可以减少一点分区数，减少最终产生的小文件数量，然后就是可以提升shuffle read的性能。

### **14. 在批处理中，如果 Driver 端 TaskScheduler 发现大量 Task 失败，重试机制是怎样的？如何区分是数据问题还是资源问题导致的反复重试？**
Spark的容错机制主要依赖于task的重试。spark有很多判断task是否失败的方式，第一时executor通过心跳机制向Driver汇报Task的状态，如果Task执行异常，或者Executor超时未汇报心跳，Driver会判定该Task失败了，然后Driver会将该Task放到调度队列重新调度，默认配置好像是允许失败4次，同时这里还有黑名单机制，为了防止坏节点导致task反复失败，失败太多的executor在会被加入黑名单，后续task不会调度到该节点。然后是如果失败了4次，spark会标记整个stage为失败，Driver会尝试重新提交stage，和task类似，如果stage失败达到上限，会标记整个作业都失败。
至于区分问题的方式，一般都还是看Spark WEB UI的指标，数据问题的话一般会表现为特定的task失败，且失败具备确定性，不管重试多少次，只要处理到该数据，就会报错，具体表现一般有数据倾斜，大部分task都完成了，极少数task还在运行，然后是特定脏数据异常，只有处理到某条数据才报错，报错信息也包含具体的业务逻辑异常
资源问题一般表现为随机性失败，失败与具体的数据内容无关，一般就是OOM，这种就是数据太大装不下或者大量数据被塞到一个executor里面了，然后是报错信息有什么lost之类的，一般就意味着机器宕机，网络抖动、gc超时之类的。